In [1]:
import pandas as pd
import ast


In [2]:
df=pd.read_csv('final_merged/tig_final.tsv', sep='\t')


In [11]:
# Potato annotation results using three annotators looks like below. User is actual user names of the annotators
# each emotion has corosponding smotion intensity scale ranging from 1 - 3, 1-low, 2-medium, 3-high, and 0/nothing intensity for neutral(no emotion) class
df = df.loc[:, df.columns != 'user']
df[1:3]


,id,text,emotion:::anger,emotion:::joy,emotion:::neutral,emotion:::surprise,emotion:::fear,emotion:::sadness,emotion:::disgust,anger:::scale_2,...,joy:::scale_1,surprise:::scale_2,surprise:::scale_3,surprise:::scale_1,sadness:::scale_2,sadness:::scale_3,sadness:::scale_1,fear:::scale_2,fear:::scale_3,fear:::scale_1
1,1632564312567869442,ዝገርም ዘተዓዛዝብ ትዕዝብቲ ?,NaN,NaN,NaN,6.0,NaN,4.0,NaN,NaN,...,NaN,su2,NaN,NaN,NaN,NaN,sa1,NaN,NaN,NaN
2,1590820123233091584,መልአክቲ ን በራዩ ህግደፍን ተደናገጽቶምን! ሴሜን ኣሜሪካ ይኣክል ጨንፈር...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df.columns

Index(['id', 'text', 'emotion:::anger', 'emotion:::joy', 'emotion:::neutral',
       'emotion:::surprise', 'emotion:::fear', 'emotion:::sadness',
       'emotion:::disgust', 'anger:::scale_2', 'anger:::scale_3',
       'anger:::scale_1', 'disgust:::scale_2', 'disgust:::scale_3',
       'disgust:::scale_1', 'joy:::scale_2', 'joy:::scale_3', 'joy:::scale_1',
       'surprise:::scale_2', 'surprise:::scale_3', 'surprise:::scale_1',
       'sadness:::scale_2', 'sadness:::scale_3', 'sadness:::scale_1',
       'fear:::scale_2', 'fear:::scale_3', 'fear:::scale_1'],
      dtype='object')

### Multi-label agreement in 3 annotators 
### using MLA (multi-label agreement)
### Paper link: https://link.springer.com/chapter/10.1007/978-3-031-44696-2_3

In [49]:
# Group by id and text
grouped = df.groupby(['id', 'text'])

# Initialize lists to store annotations for each user
annotations = []

# Iterate over groups and collect annotations
for (instance_id, displayed_text), group in grouped:
    annotation = {}
    annotation['id'] = instance_id
    annotation['text'] = displayed_text
    users = group['user'].tolist()
    for i, (_, row) in enumerate(group.iterrows()):
        user_annotations = []
        for col in row.index[3:10]:  # Iterate over columns starting from column id 3 
            if pd.notnull(row[col]):
                user_annotations.append(row[col])
        annotation[f'user{i+1}'] = row['user']#to print annotators' username
        annotation[f'anno_{i+1}'] = user_annotations
    annotations.append(annotation)

# Create DataFrame from collected annotations
df = pd.DataFrame(annotations)
df = df[['id', 'text', 'anno_1', 'anno_2', 'anno_3']]#'user1', 'user2', 'user3',
df[:5]#the annotation result is numbers from 1-7, 1=anger, 2=disgust, 3=fear, 4=sadness, 5=joy, 6=surprise, and 7=neutral

,id,text,anno_1,anno_2,anno_3
0,1,ኣብ ዞባ ደቡብ ብዝተፈጠረ ግጭት ዝተጠርጠሩ ሰባት ኣብ ትሕቲ ቀይዲ ኣትዮም።,[7.0],[7.0],[7.0]
1,2,ቢቄላ ዝብል ስም ከመይ ጌረ ይጽሕፎ?,[7.0],[7.0],[7.0]
2,3,መጀመርያ እግዚኣብሔር ሰማይን ምድርን ፈጠረ። ምድሪ ድማ ቅርጺ ዘይብላን ...,[7.0],[7.0],[7.0]
3,4,ሞት ናይዚ ግብረሽበራዊ እቲ ዝዓበየ ዓስቢ እዩ።,[2.0],"[1.0, 2.0]",[2.0]
4,5,ስም ፈጣሪ ብውሕሉል ኣገባብ ዝጥቀም ሰብ ከመይ ኢሉ የሕርቐኒ,[1.0],[7.0],[6.0]


In [50]:
df = df[['id','anno_1','anno_2','anno_3']]#make randmly shefle
# df['instance_id'] = df.reset_index().index + 1

In [51]:

df['anno_1'] = df['anno_1'].apply(lambda x: [int(y) for y in ast.literal_eval(str(x))])
df['anno_2'] = df['anno_2'].apply(lambda x: [int(y) for y in ast.literal_eval(str(x))])
df['anno_3'] = df['anno_3'].apply(lambda x: [int(y) for y in ast.literal_eval(str(x))])

In [52]:
df[:5]

,id,anno_1,anno_2,anno_3
0,1,[7],[7],[7]
1,2,[7],[7],[7]
2,3,[7],[7],[7]
3,4,[2],"[1, 2]",[2]
4,5,[1],[7],[6]


In [53]:
def calculate_a(value, coder_02_values, coder_03_values):
    if value in coder_02_values and value in coder_03_values:
        return 2
    elif value in coder_02_values or value in coder_03_values:
        return 1
    else:
        return 0

# Iterate through each row of the DataFrame
for index, row in df.iterrows():
    coder_01_values = row['anno_1']
    coder_02_values = row['anno_2']
    coder_03_values = row['anno_3']
    
    mla_po_01 = sum(
        calculate_a(value, coder_02_values, coder_03_values)
        for value in coder_01_values
    )
    
    mla_po_02 = sum(
        calculate_a(value, coder_01_values, coder_03_values)
        for value in coder_02_values
    )
    
    mla_po_03 = sum(
        calculate_a(value, coder_01_values, coder_02_values)
        for value in coder_03_values
    )
    # print(f"mla_po_01: {mla_po_01}, mla_po_02: {mla_po_02}, mla_po_03: {mla_po_03}")

    # Calculate the lengths of coder_01, coder_02, and coder_03 values
    coder_01_length = len(coder_01_values)
    coder_02_length = len(coder_02_values)
    coder_03_length = len(coder_03_values)
    
    # Calculate mla_po values
    mla_po_01 /= coder_01_length
    mla_po_02 /= coder_02_length
    mla_po_03 /= coder_03_length
    
    # Calculate total mla_po for the row
    mla_po = (mla_po_01 + mla_po_02 + mla_po_03)/6
    
    df.at[index, 'mla_po'] = mla_po


In [55]:
df[:5]

,id,anno_1,anno_2,anno_3,mla_po
0,1,[7],[7],[7],1.000000
1,2,[7],[7],[7],1.000000
2,3,[7],[7],[7],1.000000
3,4,[2],"[1, 2]",[2],0.833333
4,5,[1],[7],[6],0.000000


In [56]:
mla_po_avg = df['mla_po'].mean()
mla_po_avg

0.5194801141430355

### Pair-wise Cohens Kappa agreement

In [57]:
# Function to check if any element in list1 matches with any element in list2
def check_agreement(list1, list2):
    for item1 in list1:
        for item2 in list2:
            if item1 in list2:
                return True
    return False

# Count the number of rows where 'annotation1' and 'annotation2' agree
agreement_count1 = sum(df.apply(lambda row: check_agreement(row['anno_1'], row['anno_2']), axis=1))
agreement_count2 = sum(df.apply(lambda row: check_agreement(row['anno_2'], row['anno_3']), axis=1))
agreement_count3 = sum(df.apply(lambda row: check_agreement(row['anno_1'], row['anno_3']), axis=1))

pair_wise_kapa = (agreement_count1 / len(df) + agreement_count2 / len(df) + agreement_count3 / len(df))/3
print("pair_wise_kapa = ", pair_wise_kapa)


pair_wise_kapa =  0.573729266987694
